In [1]:
import io
import cv2
from PIL import Image
from ipywidgets import interact, IntSlider, Layout
from IPython.display import Image as IPythonImage, display

In [2]:
def get_specific_frame(video, frame_index):
    video.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    
    ret, frame = video.read()
    #video.release()
    
    if ret:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    else:
        print(f"Could not read frame at index {frame_index}")
        return None

In [3]:
def get_image(rgb_array):
    pil_img = Image.fromarray(rgb_array)

    img_byte_arr = io.BytesIO()
    pil_img.save(img_byte_arr, format='PNG')
    img_byte_arr = img_byte_arr.getvalue()
    
    return IPythonImage(data=img_byte_arr)

In [4]:
video_path = "left_eye_video.mp4"

crop_x_start = 0
crop_x_stop = 750
crop_y_start = 75
crop_y_stop = 700

In [5]:
vid = cv2.VideoCapture(video_path)

if not vid.isOpened():
    print("Error: Could not open video.")
    exit()    

def update(current_frame=431, overlay_opacity=1, kernel_size=3, iterations=1):
    frame = get_specific_frame(vid, current_frame)
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) #Maybe just make the output a grayscale originally
    frame = cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
    frame = frame[crop_y_start:crop_y_stop, crop_x_start:crop_x_stop]

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    
    _, pupil_mask = cv2.threshold(frame, 50, 255, cv2.THRESH_BINARY_INV)
    clean_pupil = cv2.morphologyEx(pupil_mask, cv2.MORPH_CLOSE, kernel, iterations=iterations)
    contours, _ = cv2.findContours(clean_pupil, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return
    
    largest_contour = max(contours, key=cv2.contourArea)

    pupil_ellipse = cv2.fitEllipseDirect(largest_contour)
    (cx, cy), (_, _), _ = pupil_ellipse

    frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)

    cv2.ellipse(frame, pupil_ellipse, (0, 255, 0), 2) 
    cv2.circle(frame, (int(cx), int(cy)), 3, (0, 0, 255), -1)

    clean_pupil = cv2.cvtColor(clean_pupil, cv2.COLOR_GRAY2RGB)

    final = cv2.addWeighted(frame, 1, clean_pupil, overlay_opacity, 1)
    
    display(get_image(final))

interact(
    update,
    current_frame=IntSlider(min=0, max=1270, step=1, value=431, layout=Layout(width='70%')),
    overlay_opacity=(0, 1, 0.1),
    kernel_size=(1, 31, 2),
    iterations=(1, 10, 1),
)

interactive(children=(IntSlider(value=431, description='current_frame', layout=Layout(width='70%'), max=1270),…

<function __main__.update(current_frame=431, overlay_opacity=1, kernel_size=3, iterations=1)>